In [1]:
# nSTAT-python notebook example: PSTHEstimation
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
SRC_PATH = (REPO_ROOT / "src").resolve()
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from scipy.io import loadmat

from nstat.data_manager import ensure_example_data
from nstat.notebook_figures import FigureTracker

np.random.seed(0)
DATA_DIR = ensure_example_data(download=True)
OUTPUT_ROOT = REPO_ROOT / "output" / "notebook_images"
__tracker = FigureTracker(topic='PSTHEstimation', output_root=OUTPUT_ROOT, expected_count=2)

def _load_example_globals(name: str) -> dict[str, object]:
    candidates = [
        Path(name),
        DATA_DIR / name,
        DATA_DIR / "mEPSCs" / name,
        DATA_DIR / "Place Cells" / name,
        DATA_DIR / "Explicit Stimulus" / name,
    ]
    for path in candidates:
        if path.exists():
            data = loadmat(path)
            return {k: v for k, v in data.items() if not k.startswith("__")}
    return {}

# SECTION 0: Section 0
# PSTH Estimation
# We illustrate two ways to estimate a peristimulus time histogram using the nSTAT toolbox. One technique is the standard binning in time, averaging across trials, and dividing by the binwidth to estimate the spike rate and the other is based on the method presented in "Analysis of Between-Trial and Within-Trial Neural Spiking Dynamics" by Czanner et al in J Neurophysiology 2008.

In [2]:
# SECTION 1: Generate a known Conditional Intensity Function
# We generated a known conditional intensity function (rate function) and generate distinct realizations of point processes consistent with this rate function. We use the method of thinning to simulate a point process.
plt.close("all")
delta = 0.001
Tmax = 10
f = .2
numRealizations = 20
__tracker.new_figure('spikeColl.plot')
__tracker.annotate('lambda.plot')

In [3]:
# SECTION 2: Estimate the PSTH with 500ms windows
__tracker.new_figure('figure')
__tracker.new_figure('figure;')
binsize = .5
__tracker.annotate('h1=trueRate.plot')
__tracker.annotate("h3=psthGLM.plot([],{{' ''k'',''Linewidth'',4'}})")
__tracker.annotate("h2=psth.plot([],{{' ''rx'',''Linewidth'',4'}})")
#
# Scalar summaries for automated parity checks.
#
# Because currently the psthGLM estimated the psth coefficients in each bin
# for each realization, we want the show the mean and standard error of the
# cofficient in each bin. We make the upper and lower confidence bounds
# equal to 1/sqrt(numRealization)=1/sqrt(psth.dimension) to view the
# standard error instead of the standard deviation
# Note the mean of the PSTH estimated via the GLM model and the PSTH computed via standard methods agree precisely. The benefit of the GLM estimated PSTH is the presence of confidence bounds on the estimate. Both the standard and GLM PSTH are in close agreement with the "true" underlying rate function (conditional intensity function) used in this simulated example. Both the PSTH and PSTHGLM code could be updated in the future to allow for variable bin sizes (e.g. in the vein of Baysian Adaptive Regression Splines by Wallstrom, Leibner and Kass). Alternatively, porting of BARS to Matlab may allow for it to be easily integrated into the nSTAT toolbox.
__tracker.finalize()